# DS2002 · Cleaning Clinic

**Studio — 2026-09-23 · Fall 2026**  
**Class time:** 45 minutes
**Name:** Simi Chakravarty

---

## Write the pipeline, then defend it

Monday I made the cleaning decisions and told you what they were. Today you make them, and the output is two things: a clean frame, and a **decision log** that says what you did to whose rows and why.

The log is not paperwork. On the midterm your team will disagree about whether a refund counts, and the log is what turns that into a two-minute conversation instead of an afternoon of re-deriving numbers.

Every step below follows the same three-part shape: **do it, count what you changed, log the decision.**

In [95]:
import pandas as pd, numpy as np
from io import StringIO
raw = '''order_id,item,category,qty,price,ts
1,Cheeseburger,Food,2,$7.50,2026-09-05T12:03:00
1,Cheeseburger,Food,2,$7.50,2026-09-05T12:03:00
2,cheese burger,food,1,7.5,09/05/2026 12:40
3,Foam Finger,Merch,NULL,12,2026-09-05 13:00:00
4,UVA T-Shirt ,Apparel,2,$24.00,2026-09-05 13:05
5,Rain Poncho,RainGear,-3,6,2026-09-05T13:20:00
6,rain poncho,rain-gear,4,$6.00,
7,,Merch,1,12,2026-09-05T14:00:00'''
df = pd.read_csv(StringIO(raw))
df

,order_id,item,category,qty,price,ts
0,1,Cheeseburger,Food,2.0,$7.50,2026-09-05T12:03:00
1,1,Cheeseburger,Food,2.0,$7.50,2026-09-05T12:03:00
2,2,cheese burger,food,1.0,7.5,09/05/2026 12:40
3,3,Foam Finger,Merch,NaN,12,2026-09-05 13:00:00
4,4,UVA T-Shirt,Apparel,2.0,$24.00,2026-09-05 13:05
5,5,Rain Poncho,RainGear,-3.0,6,2026-09-05T13:20:00
6,6,rain poncho,rain-gear,4.0,$6.00,NaN
7,7,NaN,Merch,1.0,12,2026-09-05T14:00:00


### Set up the log

Run this first. Each step calls `log()` with what happened and how many rows it touched.

In [96]:
DECISIONS = []

def log(step, decision, rows_affected):
    DECISIONS.append({'step': step, 'decision': decision, 'rows': rows_affected})
    print(f'[{step}] {decision} ({rows_affected} row(s))')

def show_log():
    return pd.DataFrame(DECISIONS)

raw_rows = len(df)
print('starting with', raw_rows, 'rows')

starting with 8 rows


### Step 0 — take inventory

**TODO:** print the shape, the dtypes, the null count per column, and the number of exact duplicate rows. Do not skip this — the rest of the studio depends on knowing what you have.

In [97]:
print(f'Shape of Dataset: {df.shape} \n')
print(f'Data Types: {df.dtypes}\n')
print(f'Null Count per Column: {df.isnull().sum()}\n')
print(f'Number of Duplicate Rows: {df.duplicated().sum()}')

Shape of Dataset: (8, 6) 

Data Types: order_id      int64
item         object
category     object
qty         float64
price        object
ts           object
dtype: object

Null Count per Column: order_id    0
item        1
category    0
qty         1
price       0
ts          1
dtype: int64

Number of Duplicate Rows: 1


**What is wrong with this data?** List at least five specific problems:

1. _..._
2. _..._
3. _..._
4. _..._
5. _..._

### Step 1 — duplicates

**TODO:** drop exact duplicate rows into a new frame called `clean`, then log how many you removed. Use `.copy()` so later assignments do not warn.

In [98]:
removed = 1   # TODO: how many duplicates were there?
clean = df.drop_duplicates().copy()  # TODO: df with duplicates dropped, copied

log('duplicates', 'dropped exact duplicate rows', removed)

[duplicates] dropped exact duplicate rows (1 row(s))


### Step 2 — price into a real number

**TODO:** strip the dollar signs and any stray whitespace, then convert to float. Assert the dtype afterward so you find out now if a stray character survived.

In [99]:
clean['price'] = pd.to_numeric(
    clean['price'].astype(str).str.replace('$', '', regex=False).str.replace(',', '', regex=False)
)

assert clean['price'].dtype == float
log('price into a real number', 'coerced to float', (raw_rows - len(clean)))

[price into a real number] coerced to float (1 row(s))


### Step 3 — quantity, and two decisions

**TODO:** coerce `qty` to numeric. Then decide, separately:

- what to do with the row that has no quantity
- what to do with the refund (negative quantity)

Log each decision with its row count. There is no single right answer — there is only an answer you can defend.

In [100]:
clean['qty'] = pd.to_numeric(clean['qty'])

missing = len(clean['qty'].isnull())    # TODO: count of NaN quantities
negative = len(clean[clean['qty'] < 0])   # TODO: count of negative quantities

dropped_missing = clean['qty'].isna().sum()
dropped_negative = (clean['qty'] < 0).sum()

clean = clean[clean['qty'].notna() & (clean['qty'] > 0)].copy()
clean['qty'] = clean['qty'].astype(int)

log('quantity, and two decisions', 'removed missing and negative values', (raw_rows - len(clean)))

[quantity, and two decisions] removed missing and negative values (3 row(s))


### Step 4 — categories that mean one thing

**TODO:** normalize case and punctuation, then map the remaining variants with an explicit dict. Print the unique values before and after so the collapse is visible. Log how many distinct categories you started and ended with.

In [101]:
print('before:', sorted(clean['category'].unique()))

# TODO: lowercase, strip, remove punctuation
clean['category'] = clean['category'].str.lower().str.strip().str.replace(r'[^\w\s]', '', regex=True)
# TODO: CATEGORY_MAP = {...} for the judgment calls
CATEGORY_MAP = {
    'food': 'food',
    'apparel': 'apparel',
    'merch': 'merch',
    'rain gear': 'rain gear',}

print('after: ', sorted(clean['category'].unique()))

before: ['Apparel', 'Food', 'Merch', 'food', 'rain-gear']
after:  ['apparel', 'food', 'merch', 'raingear']


### Step 5 — item names

**TODO:** same treatment for `item`. One product is spelled two ways, and one row has no item at all — decide what to do with it.

In [102]:
# TODO
print('before:', clean['item'].unique())
clean = clean.dropna(subset=['item'])
clean['item'] = clean['item'].str.lower().str.strip().str.replace(r'[^\w\s]', '', regex=True)

print('after: ', clean['item'].unique())

before: ['Cheeseburger' 'cheese burger' 'UVA T-Shirt ' 'rain poncho' nan]
after:  ['cheeseburger' 'cheese burger' 'uva tshirt' 'rain poncho']


### Step 6 — timestamps

**TODO:** parse `ts` into real datetimes, coercing failures to `NaT`. Report how many failed. Then add an `hour` column, which is only possible once the column is a real datetime.

In [103]:
clean['ts'] = pd.to_datetime(clean['ts'], errors='coerce')
failed = len(clean[clean['ts'].isnull()])

clean['hour'] = clean['ts'].dt.hour

### Step 7 — prove it

**TODO:** write at least five assertions that would catch a regression in this pipeline. Then compute `revenue` and print the totals.

In [110]:
# TODO: assertions

clean['revenue'] = clean['qty'] * clean['price']
# TODO: print rows, units, revenue, distinct categories
print('rows:', len(clean))
print('units:', clean['qty'].sum())
print('revenue:', clean['revenue'].sum())
print('distinct categories:', len(clean['category'].unique()))

rows: 4
units: 9
revenue: 94.5
distinct categories: 3


### Step 8 — the decision log

**TODO:** print your log. Then answer, in the markdown cell below: which single decision moved your revenue total the most, and what is the number both ways?

In [105]:
show_log()

,step,decision,rows
0,duplicates,dropped exact duplicate rows,1
1,price into a real number,coerced to float,1
2,"quantity, and two decisions",removed missing and negative values,3


In [108]:
df_clean = df.dropna().copy()

df_clean['price'] = (
    df_clean['price']
    .str.replace('$', '', regex=False)
    .astype(float)
)

df_clean['revenue'] = df_clean['qty'] * df_clean['price']

print('rows:', len(df_clean))
print('units:', df_clean['qty'].sum())
print('revenue:', df_clean['revenue'].sum())
print('distinct categories:', len(df_clean['category'].unique()))

rows: 5
units: 4.0
revenue: 67.5
distinct categories: 4


**The decision that mattered most:** _..._

**Revenue with it:** _..._  **Revenue without it:** _..._

---

## Checkpoint (participation)

Report your row count and revenue after cleaning, and the one decision that moved the total most.

Work in groups if you wish, then fill in the cell below **yourself**. Paste the printed output (or a screenshot of it) into this week's **Studio Checkpoint** in Canvas by **Thursday 11:59pm ET**. One submission per person, not per group.

In [111]:
# Checkpoint
rows_after = 4            # TODO
revenue_after = 94.5         # TODO
biggest_decision = 'The quantity, and two decisions moved the number of rows the most. It removed 3 rows.'    # TODO: which choice moved the number most
revenue_other_way = 67.5     # TODO: the total if you had chosen differently

print('rows after cleaning:', rows_after)
print('revenue:', revenue_after)
print('decision that mattered:', biggest_decision)
print('revenue the other way:', revenue_other_way)

rows after cleaning: 4
revenue: 94.5
decision that mattered: The quantity, and two decisions moved the number of rows the most. It removed 3 rows.
revenue the other way: 67.5
